# Customer Analysis

## Data Loading and Grain Collapse
We collapse the `master_table.csv` from the item grain to the order grain to ensure that order-level metrics like `payment_value` and `review_score` are not double-counted for multi-item orders.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/master_table.csv')
print('Row count before collapse:', len(df))

# Collapse to order grain
df_orders = df.drop_duplicates(subset=['order_id']).copy()
print('Row count after collapse:', len(df_orders))

Row count before collapse: 112650


Row count after collapse: 98666


## RFM Metrics Calculation
We compute Recency, Frequency, and Monetary values per `customer_unique_id`.

In [2]:
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
max_date = df_orders['order_purchase_timestamp'].max()

rfm = df_orders.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (max_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

rfm.head()

,customer_unique_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,115,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,118,1,27.19
2,0000f46a3911fa3c0805444483337064,541,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,325,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,292,1,196.89


## RFM Tiering
We split each of the RFM metrics into 3 tiers (Low, Medium, High).

In [3]:
rfm['R_tier'] = pd.qcut(rfm['recency'], 3, labels=['High', 'Medium', 'Low'])
rfm['M_tier'] = pd.qcut(rfm['monetary'], 3, labels=['Low', 'Medium', 'High'])

def f_tier(x):
    if x == 1: return 'Low'
    elif x == 2: return 'Medium'
    else: return 'High'
rfm['F_tier'] = rfm['frequency'].apply(f_tier)

print('--- Monetary Tiers ---')
print(rfm.groupby('M_tier', observed=True).agg(customer_count=('customer_unique_id', 'count'), avg_monetary=('monetary', 'mean')))

print('\n--- Recency Tiers ---')
print(rfm.groupby('R_tier', observed=True).agg(customer_count=('customer_unique_id', 'count'), avg_monetary=('monetary', 'mean')))

print('\n--- Frequency Tiers ---')
print(rfm.groupby('F_tier', observed=True).agg(customer_count=('customer_unique_id', 'count'), avg_monetary=('monetary', 'mean')))


--- Monetary Tiers ---
        customer_count  avg_monetary
M_tier                              
Low              31811     49.971718
Medium           31803    109.914297
High             31806    338.333190

--- Recency Tiers ---
        customer_count  avg_monetary
R_tier                              
High             31938    170.875829
Medium           31824    159.792310
Low              31658    167.528507

--- Frequency Tiers ---
        customer_count  avg_monetary
F_tier                              
High               240    502.421667
Low              92507    161.519281
Medium            2673    293.316423


## Summary Findings
- **Recency**: The customer base is well distributed across recent and older purchase cohorts, with average monetary value remaining fairly stable regardless of recency.
- **Frequency**: The overwhelming majority of customers made only a single purchase, showing that retention is a key area for potential growth.
- **Monetary**: A distinct high-value tier of customers contributes significantly to the overall revenue, highlighting the importance of VIP customer segments.